# Phantom noise encoding: setup

Run this notebook from the project folder, with `requirements.txt` beside it.
Run the installation cell once per environment. If packages were upgraded, restart the kernel before continuing.

For collaborators: `%pip` installs the Python libraries into the active notebook kernel.
Download the Kaggle dataset separately and extract it into `data/decoding-phantom-sounds/` beside this notebook, keeping the original folders.


In [ ]:
%pip install -r requirements.txt
%pip install xgboost


In [ ]:
# Import libraries
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


## Dataset paths

The archive has nested `Training/Training` and `Test/Test` folders.


In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "decoding-phantom-sounds"

TRAIN_DIR = DATA_DIR / "Training" / "Training"
TEST_DIR = DATA_DIR / "Test" / "Test"
LABELS_PATH = DATA_DIR / "tinnitus_train_upload.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

for path in (TRAIN_DIR, TEST_DIR, LABELS_PATH, SAMPLE_SUBMISSION_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing dataset path: {path}. Check the extraction location and notebook working directory."
        )

print(f"Dataset found: {DATA_DIR}")


## Load labels and submission format

`hasTinnitus` is the target column. The sample submission provides the expected output format.


In [ ]:
labels = pd.read_csv(LABELS_PATH, dtype={"kaggle_id": str})
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH, dtype={"kaggle_id": str})

print(f"Training labels: {labels.shape}")
print(f"Sample submission: {sample_submission.shape}")
labels.head()


## Load one EEG feature file

Start with one participant's frequency-band power features. The first CSV column contains feature names, so we use it as the index.


In [ ]:
participant_id = labels.loc[0, "kaggle_id"]
FEATURE_PATH = (
    TRAIN_DIR / participant_id
    / "features_pow_freq_bands_rest"
    / "features_pow_freq_bands_rest.csv"
)

example_power = pd.read_csv(FEATURE_PATH, index_col=0)

print(f"Participant: {participant_id}")
print(f"Power feature table: {example_power.shape}")
example_power.head()


## Match labels to participant folders

We join on `kaggle_id`, never on file order. Label rows without a training folder
are reported and excluded from this baseline; the original labels stay unchanged.
The test participants must be separate from all labeled participants.


In [ ]:
RANDOM_STATE = 42
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

if labels["kaggle_id"].isna().any() or labels["kaggle_id"].duplicated().any():
    raise ValueError("Training IDs must be present and unique.")
if not labels["hasTinnitus"].isin([0, 1]).all():
    raise ValueError("Training targets must be 0 or 1.")
if sample_submission["kaggle_id"].isna().any() or sample_submission["kaggle_id"].duplicated().any():
    raise ValueError("Test IDs must be present and unique.")

train_folder_ids = {p.name for p in TRAIN_DIR.iterdir() if p.is_dir()}
test_folder_ids = {p.name for p in TEST_DIR.iterdir() if p.is_dir()}
label_ids = set(labels["kaggle_id"])
test_ids = sample_submission["kaggle_id"].tolist()

if label_ids & test_folder_ids or train_folder_ids & test_folder_ids:
    raise ValueError("Labeled/training participants overlap the test set.")
if train_folder_ids - label_ids:
    raise ValueError("Some training folders have no matching label.")
if set(test_ids) != test_folder_ids:
    raise ValueError("Test folders do not match sample submission IDs.")

missing_training_files = labels.loc[~labels["kaggle_id"].isin(train_folder_ids)].copy()
missing_training_files.to_csv(OUTPUT_DIR / "missing_training_files.csv", index=False)
available_labels = labels.loc[labels["kaggle_id"].isin(train_folder_ids)].copy()
available_labels = available_labels.set_index("kaggle_id")

print(f"Label rows: {len(labels)}; usable training participants: {len(available_labels)}")
print(f"Excluded label rows without folders: {len(missing_training_files)}")
print(missing_training_files.to_string(index=False))
print(f"Test participants: {len(test_ids)}")


## Build one row per participant

This first model uses **frequency-band power only**. For each named channel/band
feature, calculate its mean and population standard deviation across the available
columns. This gives a fixed set of summary features even when recordings have
different numbers of columns. Channel names are aligned by name, not row position.

Only rows starting with `power_ch_` enter the model. Metadata rows (`eyes`, `site`,
`subject_id`) and label-table fields (`kaggle_id`, `center`) are excluded.
Missing channels remain missing until the model's imputation step.


In [ ]:
def summarize_power(split_dir, participant_id):
    feature_path = (
        split_dir / participant_id
        / "features_pow_freq_bands_rest"
        / "features_pow_freq_bands_rest.csv"
    )
    table = pd.read_csv(feature_path, index_col=0)
    power = table.loc[table.index.astype(str).str.startswith("power_ch_")]
    if power.empty or power.index.duplicated().any():
        raise ValueError(f"Missing or duplicate power feature names: {feature_path}")
    # Fail on unexpected text rather than silently turning it into missing data.
    power = power.apply(pd.to_numeric, errors="raise")
    power = power.replace([np.inf, -np.inf], np.nan)
    if not power.notna().any().any():
        raise ValueError(f"No usable power measurements: {feature_path}")
    summary = pd.concat([
        power.mean(axis=1).add_suffix("__mean"),
        power.std(axis=1, ddof=0).add_suffix("__std"),
    ])
    summary.name = participant_id
    return summary


def build_feature_table(split_dir, participant_ids):
    rows = []
    for number, participant_id in enumerate(participant_ids, start=1):
        rows.append(summarize_power(split_dir, participant_id))
        if number % 100 == 0 or number == len(participant_ids):
            print(f"Loaded {number}/{len(participant_ids)} participants from {split_dir.name}", flush=True)
    table = pd.DataFrame(rows).sort_index(axis=1)
    table.index.name = "kaggle_id"
    return table


X = build_feature_table(TRAIN_DIR, available_labels.index.tolist())
y = available_labels.loc[X.index, "hasTinnitus"].astype(int)
if not X.index.is_unique or not X.index.equals(y.index):
    raise ValueError("Features and targets are not aligned by participant.")

print(f"Training table: {X.shape[0]} participants x {X.shape[1]} features")
print(f"Missing feature cells: {X.isna().to_numpy().mean():.1%}")
print("Target counts:")
print(y.value_counts().sort_index())
X.iloc[:5, :6]


## Reserve participants for validation

Keep 20% of participants aside, stratified by `hasTinnitus`. Each participant has
one row, so none of their recording columns can appear in both groups. Remove
features that have no values in the fitting group using that group alone.

The median imputer is inside the model pipeline: it learns only from the fitting
group. This follows scikit-learn's [guidance on preventing data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE,
)
validation_columns = X_train.columns[X_train.notna().any(axis=0)]
X_train = X_train.loc[:, validation_columns]
X_valid = X_valid.reindex(columns=validation_columns)
assert set(X_train.index).isdisjoint(X_valid.index)

print(f"Fitting participants: {len(X_train)}")
print(f"Validation participants: {len(X_valid)}")
print(f"Features available in fitting group: {len(validation_columns)}")


## Fit a Random Forest baseline

Use 300 trees and at least three participants per leaf as a fixed starting point.
A median imputer fills missing channel features. No scaling is needed here.
We also fit a majority-class model to check whether the forest improves over a
simple constant prediction. These parameters are not tuned on validation results.


In [ ]:
forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
forest.fit(X_train, y_train)

majority = DummyClassifier(strategy="most_frequent")
majority.fit(np.zeros((len(y_train), 1)), y_train)
print("Random Forest and majority-class baseline fitted.")


## Evaluate on the held-out participants

Accuracy is the proportion correct. Balanced accuracy averages recall for the two
classes. ROC AUC measures how well scores rank the classes (0.5 is chance-level
ranking). Compare both models on exactly the same participants.

This is one validation split, not a final research estimate. Centers can appear
in both groups, and channel availability may reflect acquisition sites. Repeated
and center-held-out evaluation are needed to assess robustness to new centers.


In [ ]:
valid_predictions = forest.predict(X_valid)
positive_class_index = list(forest.classes_).index(1)
valid_probabilities = forest.predict_proba(X_valid)[:, positive_class_index]
majority_predictions = majority.predict(np.zeros((len(y_valid), 1)))
majority_probabilities = majority.predict_proba(np.zeros((len(y_valid), 1)))[:, list(majority.classes_).index(1)]


def score_predictions(name, truth, predictions, probabilities):
    return {
        "model": name,
        "accuracy": accuracy_score(truth, predictions),
        "balanced_accuracy": balanced_accuracy_score(truth, predictions),
        "roc_auc": roc_auc_score(truth, probabilities),
    }


validation_scores = pd.DataFrame([
    score_predictions("Random Forest", y_valid, valid_predictions, valid_probabilities),
    score_predictions("Majority class", y_valid, majority_predictions, majority_probabilities),
]).set_index("model")
validation_scores.to_csv(OUTPUT_DIR / "validation_scores.csv")

validation_predictions = pd.DataFrame({
    "actual_hasTinnitus": y_valid,
    "predicted_hasTinnitus": valid_predictions,
    "score_hasTinnitus": valid_probabilities,
})
validation_predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv")

print(validation_scores.round(3).to_string())
print("\nRandom Forest confusion matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_valid, valid_predictions, labels=[0, 1]),
    index=["actual_0", "actual_1"], columns=["predicted_0", "predicted_1"],
).to_string())


In [ ]:
from xgboost import XGBClassifier

xgb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=4,
    )),

])
xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_valid)
positive_index = list(xgb_model.classes_).index(1)
xgb_probabilities = xgb_model.predict_proba(X_valid)[:, positive_index]

validation_scores.loc["XGBoost"] = {
    "model": "XGBoost",
    "accuracy": accuracy_score(y_valid, xgb_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_valid, xgb_predictions),
    "roc_auc": roc_auc_score(y_valid, xgb_probabilities),
}

print("validation accuracy (%):")
print((validation_scores["accuracy"] * 100).round(1))

In [ ]:
accuracy_percent = validation_scores["accuracy"] * 100

for model__name, percentage in accuracy_percent.items():
    print(f"{model__name} accuracy: {percentage:.1f}%")
    
fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(accuracy_percent.index, accuracy_percent.values, color=["steelblue", "grey"])

ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_ylim(0, 100)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Validation Accuracy Comparison")
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

## Fit all available training participants and predict the test set

Fit a fresh copy of the same pipeline on all usable labeled participants, then
predict test IDs in the sample submission's order. Features are aligned to the
training table; test values never fit the imputer or the classifier.

The sample submission's zeros are placeholders, not known test outcomes. The
export contains model predictions with unknown test accuracy. Nothing is uploaded
to Kaggle, and the original files are unchanged.


In [ ]:
X_test = build_feature_table(TEST_DIR, test_ids)
final_columns = X.columns[X.notna().any(axis=0)]
X_full = X.loc[:, final_columns]
X_test = X_test.reindex(index=test_ids, columns=final_columns)
if X_test.isna().all(axis=1).any():
    raise ValueError("A test participant has no features matching the training schema.")

final_model = clone(forest)
final_model.fit(X_full, y)
test_predictions = final_model.predict(X_test)

submission = sample_submission.copy()
submission["hasTinnitus"] = test_predictions.astype(int)
assert submission["kaggle_id"].tolist() == test_ids
assert submission["hasTinnitus"].isin([0, 1]).all()
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)

print(f"Saved {len(submission)} predictions to {OUTPUT_DIR / 'submission.csv'}")
print("Predicted class counts:")
print(submission["hasTinnitus"].value_counts().sort_index())
submission.head()
